# 4장 단기 메모리

이전 대화를 저장하고 다시 사용하는 방법을 살펴봅니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 4장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

---

## 1 - 언어 모델 선택

이번 실습에 사용할 언어 모델을 지정합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B(내장 추론과 도구 호출 없음)
llm = LLM(model="gemma3:12b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

기본 `Memory`는 대화 이력을 계속 보관합니다. 이번 실습에서는 모델에 보내는 대화의 길이를 제한하는 두 가지 단기 메모리 방식을 구현합니다.

## 2. 최근 메시지만 보관하는 `TrimmingMemory`

대화가 길어져 모델의 컨텍스트 창에 들어가지 않으면 오래된 대화부터 제거할 수 있습니다. `Memory` 클래스를 상속하고 `add`를 수정해 최근 두 번의 문답만 남깁니다. 여기서 한 번의 문답은 사용자 메시지와 이에 대한 모델의 응답 한 쌍입니다.

2장의 에이전트는 이전 대화를 전달하지 않았습니다. 새 메모리에서는 대화 중 어느 부분이 유지되는지 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import Memory

class TrimmingMemory(Memory):
    """사용자와 모델의 최근 두 번의 문답을 보관한다."""

    def add(self, role: str, content: str, **kwargs) -> None:
        # 부모 클래스 Memory의 메서드로 새 메시지 추가
        super().add(role, content, **kwargs)

        # 시스템 메시지와 최근 두 번의 문답(메시지 네 개) 유지
        system = [message for message in self.messages if message["role"] == "system"]
        turns = [message for message in self.messages if message["role"] != "system"]
        self.messages = system + turns[-4:]

메시지를 여러 번 보내 최근 두 번의 문답만 남는지 살펴봅니다.

In [ ]:
from illustrated_agents.chapters.ch4 import TinyAgent

# 에이전트 생성
memory = TrimmingMemory()
agent = TinyAgent(llm=llm, memory=memory)

# 여러 차례 대화
response_1 = agent.run("Hi! I'm reading 'An Illustrated Guide to AI Agents'.")
response_2 = agent.run("There are many flamingos in this book, why?")
response_3 = agent.run("What is 1+1?")
response_4 = agent.run("What is 2+2?")

현재 메모리에 저장된 메시지를 출력합니다.

In [ ]:
from rich import print
print(agent.memory.get_messages())

최근 두 번의 문답만 남기면 입력 길이는 줄지만, 초반에 전달한 중요한 정보도 사라질 수 있습니다. 전체 대화는 별도로 수집한 `Trajectory`에서 확인할 수 있습니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

## 3. 대화를 요약하는 `SummarizationMemory`

대화가 한 차례 끝날 때마다 같은 모델이나 별도의 모델로 대화를 요약할 수 있습니다. 대화 전체를 다시 요약하거나 일부만 요약하는 등 여러 방법이 있습니다. 여기서는 응답에 사용한 모델로 대화 내용을 요약하고, 요약문을 `system` 메시지에 담아 `user`와 `assistant`의 실제 발화와 구분합니다.

In [ ]:
class SummarizationMemory(Memory):
    """언어 모델로 지난 대화를 요약하고 요약문을 갱신한다."""

    def __init__(self, llm: LLM):
        super().__init__()
        self.llm = llm

    def add(self, role: str, content: str, **kwargs) -> None:
        # 부모 클래스 Memory의 메서드로 새 메시지 추가
        super().add(role, content, **kwargs)

        # 문답이 끝날 때마다 누적 요약 갱신
        if role == "assistant":
            # 기존 요약과 새 대화 분리
            summary = ""
            conversation = ""
            for message in self.messages:
                if message["role"] == "system":
                    summary = message["content"]
                else:
                    conversation += f"{message['role']}: {message['content']}\n"

            # 기존 요약에 새 대화를 반영하도록 모델에 요청
            prompt = f"""Update the summary with the new conversation.

Summary: {summary}

Conversation:
{conversation}
Output the updated summary only."""
            response = self.llm.generate([{"role": "user", "content": prompt}])
            self.messages = [{"role": "system", "content": response.content}]

`SummarizationMemory`를 에이전트에 연결합니다.

In [ ]:
# 에이전트 생성
memory = SummarizationMemory(llm=llm)
agent = TinyAgent(llm=llm, memory=memory)

# 질문 실행
response = agent.run("Hi, my name is Sarah. Why are flamingos pink?")
print(response)

메모리에서 `system` 메시지를 열어 대화가 어떤 내용으로 요약되었는지 확인합니다.

In [ ]:
print(agent.memory.get_messages())

에이전트에 질문을 몇 번 더 보내 요약이 어떻게 달라지는지 확인합니다.

In [ ]:
response = agent.run("Tell me about yourself.")
response = agent.run("What is your favorite color?")
response = agent.run("Are you fully autonomous?")
print(agent.memory.get_messages())

요약문에 이후 질문의 내용이 추가된 것을 볼 수 있습니다. 전체 실행 과정은 아래에서 확인합니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

---

# 구현한 내용

대화 기록을 관리하는 `Memory`를 `TinyAgent`에 연결했습니다. 아래에서 언어 모델, 메모리, 에이전트의 구성을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import what_we_built_st; what_we_built_st

# 다음 실습

[장기 메모리](chapter04_long_term_memory.ipynb)에서는 검색 증강 생성으로 관련 문서를 찾습니다. [5장 도구](../chapter05/chapter05.ipynb)로 이동하면 에이전트가 도구를 호출하는 과정을 실습할 수 있습니다.